# DICE vs Surface DICE Comparison

This notebook compares DICE and Surface DICE metrics for segmentation mask evaluation.

## Objectives:
- Load ground truth and predicted masks for a specific case
- Calculate DICE scores per organ
- Calculate Surface DICE scores per organ
- Compare and visualize the results

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import evaluation functions
from radcure_processor.evaluation import SegmentationEvaluator

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Configuration

Set the paths to your data folders and specify the case ID you want to evaluate.

In [ ]:
# ============================================================================
# CONFIGURATION - Modify these paths according to your setup
# ============================================================================

# Case ID to evaluate (without .nii.gz extension)
CASE_ID = "case_0405"  # Example: "case_0405", "case_0123", etc.

# Paths to folders containing masks
LABELS_FOLDER = "/path/to/DatasetXXX_TotalSegmentator/labelsTs"  # Ground truth masks
PREDICTED_LABELS_FOLDER = "/path/to/DatasetXXX_TotalSegmentator/labelsTs_predicted"  # Predicted masks

# Path to organ dictionary JSON file
ORGAN_DICTIONARY_PATH = "/path/to/organ_dictionary.json"

# Voxel spacing in mm (x, y, z) - IMPORTANT for Surface DICE calculation
# This should match the actual spacing of your images
# Common values: (0.5, 0.5, 1.0) for CT scans, (1.0, 1.0, 1.0) for isotropic
SPACING_MM = (1.0, 1.0, 1.0)  # Modify based on your data

# Surface DICE tolerance in mm (default: 3.0 mm)
# This is the maximum distance from the surface that is considered acceptable
SURFACE_DICE_TOLERANCE_MM = 3.0

# ============================================================================
# Validate paths
# ============================================================================
gt_path = os.path.join(LABELS_FOLDER, f"{CASE_ID}.nii.gz")
pred_path = os.path.join(PREDICTED_LABELS_FOLDER, f"{CASE_ID}.nii.gz")

print(f"Case ID: {CASE_ID}")
print(f"Ground truth path: {gt_path}")
print(f"Predicted path: {pred_path}")
print(f"Organ dictionary: {ORGAN_DICTIONARY_PATH}")
print(f"Voxel spacing: {SPACING_MM} mm")
print(f"Surface DICE tolerance: {SURFACE_DICE_TOLERANCE_MM} mm")
print()

# Check if files exist
if not os.path.exists(gt_path):
    print(f"⚠️  WARNING: Ground truth file not found: {gt_path}")
if not os.path.exists(pred_path):
    print(f"⚠️  WARNING: Predicted file not found: {pred_path}")
if not os.path.exists(ORGAN_DICTIONARY_PATH):
    print(f"⚠️  WARNING: Organ dictionary not found: {ORGAN_DICTIONARY_PATH}")

## Load Masks and Calculate Metrics

We'll use the independent functions `calculate_dice` and `calculate_surface_dice` to compute both metrics.

In [ ]:
# Initialize evaluator
evaluator = SegmentationEvaluator()

# Calculate DICE scores
print("Calculating DICE scores...")
dice_scores = evaluator.calculate_dice(
    gt_mask=gt_path,
    pred_mask=pred_path,
    organ_dictionary_path=ORGAN_DICTIONARY_PATH,
    spacing_mm=SPACING_MM
)

print(f"✓ DICE calculation complete")
print(f"  Found {len([k for k in dice_scores.keys() if k != 'overall_dice'])} organs")
print(f"  Overall DICE: {dice_scores.get('overall_dice', 'N/A'):.4f}")
print()

In [ ]:
# Calculate Surface DICE scores
print("Calculating Surface DICE scores...")
try:
    surface_dice_scores = evaluator.calculate_surface_dice(
        gt_mask=gt_path,
        pred_mask=pred_path,
        organ_dictionary_path=ORGAN_DICTIONARY_PATH,
        spacing_mm=SPACING_MM,
        tolerance_mm=SURFACE_DICE_TOLERANCE_MM
    )
    print(f"✓ Surface DICE calculation complete")
    print(f"  Found {len([k for k in surface_dice_scores.keys() if k != 'overall_surface_dice'])} organs")
    print(f"  Overall Surface DICE: {surface_dice_scores.get('overall_surface_dice', 'N/A'):.4f}")
except ImportError as e:
    print(f"✗ Error: {e}")
    print("  Please install surface_distance package:")
    print("  pip install git+https://github.com/google-deepmind/surface-distance.git")
    surface_dice_scores = {}
except Exception as e:
    print(f"✗ Error calculating Surface DICE: {e}")
    surface_dice_scores = {}
print()

## Compare Results

Create a comparison table and visualizations.

In [ ]:
# Create comparison DataFrame
comparison_data = []

# Get all organ names (excluding overall metrics)
organ_names = set()
for key in dice_scores.keys():
    if key != 'overall_dice':
        organ_names.add(key)
for key in surface_dice_scores.keys():
    if key != 'overall_surface_dice':
        organ_names.add(key)

organ_names = sorted(organ_names)

for organ_name in organ_names:
    comparison_data.append({
        'Organ': organ_name,
        'DICE': dice_scores.get(organ_name, np.nan),
        'Surface DICE': surface_dice_scores.get(organ_name, np.nan)
    })

# Add overall metrics
comparison_data.append({
    'Organ': 'OVERALL',
    'DICE': dice_scores.get('overall_dice', np.nan),
    'Surface DICE': surface_dice_scores.get('overall_surface_dice', np.nan)
})

df_comparison = pd.DataFrame(comparison_data)

# Display comparison table
print("=" * 70)
print(f"Comparison: DICE vs Surface DICE for Case {CASE_ID}")
print("=" * 70)
print(df_comparison.to_string(index=False))
print("=" * 70)

In [ ]:
# Create visualization comparing DICE and Surface DICE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Filter out overall row for organ-level visualization
df_organs = df_comparison[df_comparison['Organ'] != 'OVERALL'].copy()

# Plot 1: Bar plot comparing DICE and Surface DICE per organ
x = np.arange(len(df_organs))
width = 0.35

ax1 = axes[0]
bars1 = ax1.bar(x - width/2, df_organs['DICE'], width, label='DICE', alpha=0.8)
bars2 = ax1.bar(x + width/2, df_organs['Surface DICE'], width, label='Surface DICE', alpha=0.8)

ax1.set_xlabel('Organ', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title(f'DICE vs Surface DICE Comparison\nCase: {CASE_ID}', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(df_organs['Organ'], rotation=45, ha='right')
ax1.set_ylim([0, 1.1])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=8)

# Plot 2: Scatter plot showing correlation
ax2 = axes[1]
scatter = ax2.scatter(df_organs['DICE'], df_organs['Surface DICE'], 
                     s=100, alpha=0.6, edgecolors='black', linewidth=1)

# Add diagonal line (perfect correlation)
ax2.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect correlation')

# Add organ labels
for idx, row in df_organs.iterrows():
    if not (np.isnan(row['DICE']) or np.isnan(row['Surface DICE'])):
        ax2.annotate(row['Organ'], 
                    (row['DICE'], row['Surface DICE']),
                    fontsize=8, alpha=0.7)

ax2.set_xlabel('DICE Score', fontsize=12)
ax2.set_ylabel('Surface DICE Score', fontsize=12)
ax2.set_title('DICE vs Surface DICE Correlation', fontsize=14, fontweight='bold')
ax2.set_xlim([0, 1.1])
ax2.set_ylim([0, 1.1])
ax2.legend()
ax2.grid(True, alpha=0.3)

# Calculate correlation
valid_data = df_organs.dropna(subset=['DICE', 'Surface DICE'])
if len(valid_data) > 1:
    correlation = valid_data['DICE'].corr(valid_data['Surface DICE'])
    ax2.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax2.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Calculate difference between DICE and Surface DICE
df_organs['Difference'] = df_organs['DICE'] - df_organs['Surface DICE']
df_organs['Abs_Difference'] = df_organs['Difference'].abs()

# Sort by absolute difference (largest differences first)
df_organs_sorted = df_organs.sort_values('Abs_Difference', ascending=False)

print("=" * 70)
print("Difference Analysis: DICE - Surface DICE")
print("=" * 70)
print("Positive difference = DICE > Surface DICE (volume overlap better than boundary)")
print("Negative difference = DICE < Surface DICE (boundary overlap better than volume)")
print()
print(df_organs_sorted[['Organ', 'DICE', 'Surface DICE', 'Difference']].to_string(index=False))
print("=" * 70)

# Visualize differences
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in df_organs_sorted['Difference']]
bars = ax.barh(df_organs_sorted['Organ'], df_organs_sorted['Difference'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Difference (DICE - Surface DICE)', fontsize=12)
ax.set_ylabel('Organ', fontsize=12)
ax.set_title(f'Difference Between DICE and Surface DICE\nCase: {CASE_ID}', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, df_organs_sorted['Difference'])):
    if not np.isnan(val):
        ax.text(val, i, f'{val:+.3f}', 
               va='center', ha='right' if val < 0 else 'left', fontsize=9)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Summary statistics
print("=" * 70)
print("Summary Statistics")
print("=" * 70)
print(f"Case ID: {CASE_ID}")
print(f"Number of organs evaluated: {len(df_organs)}")
print()
print("DICE Scores:")
print(f"  Mean: {df_organs['DICE'].mean():.4f}")
print(f"  Median: {df_organs['DICE'].median():.4f}")
print(f"  Std: {df_organs['DICE'].std():.4f}")
print(f"  Min: {df_organs['DICE'].min():.4f}")
print(f"  Max: {df_organs['DICE'].max():.4f}")
print()
print("Surface DICE Scores:")
print(f"  Mean: {df_organs['Surface DICE'].mean():.4f}")
print(f"  Median: {df_organs['Surface DICE'].median():.4f}")
print(f"  Std: {df_organs['Surface DICE'].std():.4f}")
print(f"  Min: {df_organs['Surface DICE'].min():.4f}")
print(f"  Max: {df_organs['Surface DICE'].max():.4f}")
print()
print("Overall Metrics:")
print(f"  Overall DICE: {dice_scores.get('overall_dice', np.nan):.4f}")
if 'overall_surface_dice' in surface_dice_scores:
    print(f"  Overall Surface DICE: {surface_dice_scores.get('overall_surface_dice', np.nan):.4f}")
print("=" * 70)